<a href="https://colab.research.google.com/github/ruso404/GEN-AI-EXPS/blob/main/PersonaForgeAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch detoxify sentencepiece -q


In [ ]:
import torch
from transformers import pipeline, AutoTokenizer
from detoxify import Detoxify

device = 0 if torch.cuda.is_available() else -1
print(f"🖥️ Using {'GPU' if device == 0 else 'CPU'}")

print("⏳ Loading models...")

# ✅ TinyLlama — small, fast, actually follows instructions
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

generator = pipeline(
    "text-generation",
    model=MODEL_NAME,
    tokenizer=tokenizer,
    device=device,
    torch_dtype=torch.float16 if device == 0 else torch.float32
)

sentiment = pipeline(
    "sentiment-analysis",
    device=device,
    truncation=True,
    max_length=512
)

toxicity_model = Detoxify('original')
print("🔥 PersonaForge AI Ready!")

🖥️ Using GPU
⏳ Loading models...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

🔥 PersonaForge AI Ready!


In [ ]:
# ============================================================
# 🔧 CELL 3 — Core Functions
# ============================================================

def generate_persona(name, profession, mood, background):
    # TinyLlama uses ChatML format
    messages = [
        {"role": "system", "content": "You are a creative character writer. Write vivid, detailed personality profiles."},
        {"role": "user", "content": (
            f"Write a personality profile for:\n"
            f"Name: {name}\n"
            f"Profession: {profession}\n"
            f"Mood: {mood}\n"
            f"Background: {background}\n"
            f"Describe their personality traits, speaking style, and behavior in 3-4 sentences."
        )}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    output = generator(prompt, max_new_tokens=150, do_sample=True, temperature=0.9,
                       pad_token_id=tokenizer.eos_token_id)
    full_text = output[0]['generated_text']
    # Extract only the assistant's reply (after the last <|assistant|> tag)
    reply = full_text.split("<|assistant|>")[-1].strip()
    return reply


def chat_with_persona(persona_description, user_input):
    messages = [
        {"role": "system", "content": f"You are roleplaying as this character:\n{persona_description}\nStay in character at all times."},
        {"role": "user", "content": user_input}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = generator(prompt, max_new_tokens=80, do_sample=True, temperature=0.8,
                         pad_token_id=tokenizer.eos_token_id)
    full_text = response[0]['generated_text']
    # Extract only the assistant's reply
    reply = full_text.split("<|assistant|>")[-1].strip()
    return reply


def analyze_text(text):
    print("\n📊 ANALYSIS REPORT:")
    sent = sentiment(text[:500])[0]
    label_emoji = "😊" if sent['label'] == "POSITIVE" else "😟"
    print(f"  Sentiment : {label_emoji} {sent['label']} (confidence: {sent['score']:.2f})")
    tox = toxicity_model.predict(text)
    tox_score = round(tox['toxicity'], 3)
    print(f"  Toxicity  : {tox_score}")
    if tox_score > 0.5:
        print("  ⚠️  Warning: High toxicity detected!")
    else:
        print("  ✅ Content appears safe.")
    print("-" * 40)

In [ ]:
print("=" * 40)
print("   🎭 PERSONAFORGE AI — CREATE YOUR PERSONA")
print("=" * 40)

name       = input("Enter Name       : ")
profession = input("Enter Profession : ")
mood       = input("Enter Mood       : ")
background = input("Enter Background : ")

print("\n⏳ Generating persona...")
persona = generate_persona(name, profession, mood, background)

print("\n🎭 GENERATED PERSONA:\n")
print(persona)
print("\n" + "=" * 40)
print("💬 Start chatting! Type 'exit' to quit.")
print("=" * 40)

# ✅ Fix 5: Clean loop structure
while True:
    try:
        user_input = input("\nYou: ").strip()
        if not user_input:
            continue
        if user_input.lower() == "exit":
            print("👋 Exiting PersonaForge. Goodbye!")
            break

        print("\n⏳ Generating response...")
        response = chat_with_persona(persona, user_input)

        print("\n🤖 Persona Response:\n")
        print(response)
        analyze_text(response)

    except KeyboardInterrupt:
        print("\n👋 Session interrupted. Goodbye!")
        break

   🎭 PERSONAFORGE AI — CREATE YOUR PERSONA
Enter Name       : John 
Enter Profession : Police Officeer
Enter Mood       : Strict
Enter Background : A strict police officer with 30 years of work experience


Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⏳ Generating persona...

🎭 GENERATED PERSONA:

John is a stern and strict police officer, always striving for perfection. They have a clear and concise mind, with a strong sense of justice and respect for the law. They are impatient when dealing with minor infractions, but remain professional when dealing with more grave offenses.

John's speaking style is direct and firm, with little chance of flipping over to a more polite or accommodating tone. They tend to speak to their subordinates in a blunt and authoritative manner, often giving them stern warnings or firm demands.

John's behavior is often marked by strict adherence to the law, with little consideration for personal preferences or desires. They are

💬 Start chatting! Type 'exit' to quit.

You: How are You officer


Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



⏳ Generating response...

🤖 Persona Response:

I do not have a specific personality of an officer like you. However, in the context of a police officer, you could describe their behavior by saying that they are strictly following the law and are rigorously enforcing it. They would be unlikely to do anything that would deviate from the established rules or procedures, even if it conflicts with their personal preferences or desires. Their behavior

📊 ANALYSIS REPORT:
  Sentiment : 😟 NEGATIVE (confidence: 0.98)
  Toxicity  : 0.004000000189989805
  ✅ Content appears safe.
----------------------------------------
